# Runnable

**Runnable 是“砖块（接口协议）”，而 LCEL 是“粘合剂（表达语言）”**。

------

### 1. Runnable：万物皆可运行的“协议”

**Runnable** 是 LangChain 定义的一个**通用接口协议**。在 LangChain 中，几乎所有的核心组件（如 Prompt、LLM、OutputParser、Retriever）都实现了这个接口。

#### 为什么需要 Runnable？

在没有 Runnable 之前，每个组件的调用方式各不相同。

以前 LLM 用 `predict()`，Retriever 用 `get_relevant_documents()`，这种不统一是开发者的噩梦。现在它们全都是 Runnable，统一用 `invoke()`。

有了 Runnable 后，所有组件都必须支持以下标准方法：

- **`invoke()`**：处理单个输入。
- **`batch()`**：并行处理多个输入。
- **`stream()`**：流式输出结果。
- **`ainvoke()` / `abatch()` / `astream()`**：对应的异步版本。

**理解直觉：** 你可以把 Runnable 想象成插头。无论这个电器是电视机还是电冰箱，只要它符合“三相插头”的标准，就可以插在插座上使用。

# LCEL (LangChain Expression Language)

是基于 Runnable 协议开发的一种**声明式语言**。它最显著的特征是使用了 Python 的 **`|` (管道符)**。

基于Runnable特性，设计实现了LCEL声明式编码方式，通过管道符链接，定义的一个chain链式，只需要在最后chain.invoke声明一下，chain中的其他Runnable对象就会按照顺序自行分别调用invoke，然后将前一步的输出作为下一步的输入。

另外为了提高效率，也可以使用使用 `RunnableParallel` 可以让多个步骤同时跑。

#### LCEL 的核心逻辑

LCEL 允许你通过 `|` 将不同的 Runnable 对象连接起来，形成一个 **RunnableSequence**。

**代码示例：**

Python

```
# 这就是 LCEL 表达式
chain = prompt | model | output_parser
```

**LCEL 的优势：**

1. **流式支持 (Streaming)**：只要链中的组件支持流式，整个链就能自动支持流式。
2. **并行执行 (Parallelism)**：使用 `RunnableParallel` 可以让多个步骤同时跑，大幅提升效率。
3. **自动重试与回退**：可以轻松给某个环节增加 `with_fallbacks`。
4. **可视化与追踪**：LCEL 编写的链在 LangSmith 中能被清晰地拆解成一个个步骤。

# 代码示例


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from deepagents.middleware.subagents import SubAgentMiddleware, SubAgent
from deepagents.backends import StateBackend
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool


# --- 1. 准备组件 (Runnable化) ---

# A. 模型
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)


@tool
def get_weather(city: str) -> str:
    """获取位置的天气信息。"""
    return f"{city} 的天气：晴朗，72°F"


# B. 带有子代理能力的 Webster Agent (它也是一个 Runnable)
weather_worker = SubAgent(
    name="weather_pro",
    description="查询天气的专家",
    system_prompt="使用 get_weather 查天气。",
    tools=[get_weather],
    model=model,
)


webster_agent = create_agent(
    model=model,
    middleware=[SubAgentMiddleware(backend=StateBackend(), subagents=[weather_worker])],
)

# C. 提示词模板 (Runnable)
prompt = ChatPromptTemplate.from_template(
    "根据以下天气信息：{weather_info}，为城市 {city} 生成一份 JSON 格式的穿衣建议。包含：'clothes' 和 'notice' 两个字段。"
)

# D. 输出解析器 (Runnable)
parser = JsonOutputParser()

# --- 2. 串联执行链 (LCEL) ---
chain = (
    # 第一步：准备数据
    # 我们用字典形式，city 直接传，weather_info 则去调用 webster_agent 获取
    {"city": RunnablePassthrough(), "weather_info": webster_agent}
    # 第二步：将准备好的字典喂给 Prompt
    | prompt
    # 第三步：Prompt 的结果喂给模型
    | model
    # 第四步：模型的结果喂给解析器
    | parser
)
# --- 3. 最终触发 ---

if __name__ == "__main__":
    # 只需要这一次 invoke，整个链条会自动：
    # 1. 启动 webster_agent 查天气
    # 2. 把天气填入 prompt
    # 3. 让模型生成建议
    # 4. 把字符串转成 Python 字典
    
    # LangGraph 的节点（Node）必须返回一个字典（dict）来更新全局状态，而你直接返回了一个字符串 "上海"
    final_result = chain.invoke({"city":"上海"})

    print(type(final_result))  # <class 'dict'>
    print(final_result)

<class 'dict'>
{'clothes': ['T恤', '薄外套'], 'notice': '由于当前气温约为22摄氏度，建议穿着轻薄的长袖上衣和一件薄外套以备不时之需。'}


# 代码解析
### 1. 数据流向

当你调用 `chain.invoke("上海")` 时，输入值 `"上海"` 就像一个包裹，进入了第一个环节。

#### **RunnablePassthrough() 是什么？**

它就像一根**透明的管子**。它的唯一作用就是：**把输入的原封不动地传下去**。

- 当 `"上海"` 经过它时，它直接输出 `"上海"`。
- 结果：`city` 这个 key 就拿到了值 `"上海"`。

#### **为什么传 Key-Value 对？给谁看？**

这是为了给**下一步（Prompt）**看的。

- 你的 `Prompt` 里面定义了变量：`{city}` 和 `{weather_info}`。
- Prompt 就像一个“模板填充机”，它需要一个 **字典** 才能知道把哪个值填到哪个坑里。
- 这一步的输出结果其实是一个字典：`{"city": "上海", "weather_info": "上海晴，25°C"}`。

#### **数据流动的真相：**

1. 你输入 `"上海"`。
2. `RunnableParallel`（即这个大括号 `{...}`）启动。
3. 它发现有两条分片：
   - 左边派 `RunnablePassthrough` 去接 `"上海"`。
   - 右边派 `webster_agent` 去拿 `"上海"` 查天气。
4. 两条路跑完，汇总成一个字典，传给后面的 `| prompt`。

### 🔍 数据流动的“显微镜”视角

当你执行 `chain.invoke({"city":"上海"})` 时，数据经历了以下四次“变身”：

1. **分流阶段 (Parallel)**：
   - `city` 拿到 `"上海"`。
   - `weather_info` 拿着整个 `{"city":"上海"}` 去找 `webster_agent`。
   - **关键点**：因为你传的是字典，`webster_agent` 内部的 LangGraph 节点终于找到了它要的 `city` 字段（或者叫 State），所以不再报 `InvalidUpdateError`。
2. **聚合阶段 (Prompting)**：
   - `prompt` 拿到了一个完整的“大礼包”：`{"city": "上海", "weather_info": "...晴朗，72°F"}`。
   - 它把这两个值精准地填进了模板里的 `{city}` 和 `{weather_info}`。
3. **生成阶段 (LLM)**：
   - `model` 接收到填充好的长字符串。注意，此时 `model` 只是一个纯粹的生成器，它不再需要考虑怎么查天气。
4. **提纯阶段 (Parsing)**：
   - `parser` 把模型吐出来的 Markdown 字符串（带 ````json` 的那种）剥离干净，转成了真正的 Python `dict`。

   
### 2. 为什么后面接 `| model` 而不是 `| webster_agent`？

这是一个非常棒的洞察！这里涉及到了 **Agent 的双重身份**。

#### **webster_agent 在这里的角色：工具人 (Information Gatherer)**

在第一步里，`webster_agent` 的作用是**提供素材**。

- 它的任务是：去查上海的天气，把原始数据带回来。
- 它是作为 `Prompt` 的一个“数据源”存在的。

#### **最后的 model 的角色：主笔人 (Final Decision Maker)**

在这一步，我们已经拿到了所有素材（城市名 + 天气情况）。现在我们需要一个“大脑”来把这些素材加工成 JSON 格式。

- **为什么不继续用 `webster_agent`？** 因为 `webster_agent` 是带中间件的，它脑子里整天想着怎么调工具、怎么派发任务。
- **为什么用 `model`？** 这里的 `model` 是纯粹的 LLM。它不需要再调工具了，它只需要**安安静静地写稿子**，根据 Prompt 生成 JSON。